##### Purpose

Perform final acceptance testing of the registered Candidate model in Unity Catalog.

The notebook validates that the governed model artifact can:

``` text

Load from Unity Catalog
        ↓
Accept correctly preprocessed test data
        ↓
Produce valid logits
        ↓
Produce valid probabilities
        ↓
Produce churn classifications
        ↓
Meet predefined acceptance criteria
        ↓
PASS / FAIL
        ↓
Promote Candidate → Champion if PASS

```

This notebook does not train or tune the model.


##### Production Design

The lifecycle is now:

``` text

07_mlflow_experiment_comparison
        ↓
Select Candidate
        ↓
Persist model + preprocessing
        ↓
08_model_registration
        ↓
Unity Catalog Version
        ↓
@Candidate
        ↓
09_final_testing_and_conclusion
        ↓
Final Acceptance Test
        ↓

      PASS?
     /     \
   YES      NO
    ↓        ↓
Champion   Remain Candidate

```

The critical principle is:

Acceptance thresholds must be defined before looking at final acceptance results.

##### 1. Load Project Configuration

In [0]:
%run ./00_project_config

##### 2. Import Required Components

In [0]:
import json
import joblib

import mlflow
import mlflow.pytorch

import numpy as np
import torch
import torch.nn as nn

from mlflow import MlflowClient

from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)

from sklearn.model_selection import (
    train_test_split,
)

##### 3. Configure Unity Catalog Registry

In [0]:
mlflow.set_registry_uri(
    "databricks-uc"
)

In [0]:
client = MlflowClient()

##### 4. Read the Selection Manifest

In [0]:
with open(
    SELECTION_MANIFEST_PATH,
    "r",
) as file:

    selection_manifest = json.load(
        file
    )

In [0]:
selection_manifest

In [0]:
SELECTED_EXPERIMENT = (
    selection_manifest[
        "selected_experiment"
    ]
)

SELECTED_RUN_ID = (
    selection_manifest[
        "selected_run_id"
    ]
)

SELECTED_MODEL_URI = (
    selection_manifest[
        "selected_model_uri"
    ]
)

PREPROCESSOR_PATH = (
    selection_manifest[
        "preprocessor_path"
    ]
)

##### 5. Resolve the Registered Candidate

In [0]:
candidate_version = (
    client.get_model_version_by_alias(
        name=REGISTERED_MODEL_NAME,
        alias=CANDIDATE_ALIAS,
    )
)

In [0]:
CANDIDATE_VERSION = str(
    candidate_version.version
)

In [0]:
print(
    "Registered model:",
    REGISTERED_MODEL_NAME,
)

print(
    "Candidate version:",
    CANDIDATE_VERSION,
)

print(
    "Source experiment:",
    SELECTED_EXPERIMENT,
)

print(
    "Source Run ID:",
    SELECTED_RUN_ID,
)

##### 6. Build Candidate Model URI

In [0]:
CANDIDATE_MODEL_URI = (
    f"models:/"
    f"{REGISTERED_MODEL_NAME}"
    f"@{CANDIDATE_ALIAS}"
)

In [0]:
print(
    "Candidate model URI:",
    CANDIDATE_MODEL_URI,
)

##### 7. Load the Registered Candidate

In [0]:
candidate_model = (
    mlflow.pytorch.load_model(
        CANDIDATE_MODEL_URI
    )
)

candidate_model.eval()

In [0]:
print(
    candidate_model
)

##### 8. Load the Exact Fitted Preprocessor

In [0]:
preprocessor = joblib.load(
    PREPROCESSOR_PATH
)

In [0]:
print(
    "Loaded preprocessor from:",
    PREPROCESSOR_PATH,
)

Training
→ fit preprocessor

Final testing
→ load exact fitted preprocessor
→ transform only

We do not call fit_transform() here.

##### 9. Load Gold Dataset

In [0]:
df = (
    spark.table(
        GOLD_TABLE
    )
    .toPandas()
)

In [0]:
y = df[
    TARGET_COLUMN
]

X = df.drop(
    columns=COLUMNS_TO_DROP,
    errors="ignore",
)

In [0]:
print(
    "X shape:",
    X.shape,
)

print(
    "y shape:",
    y.shape,
)

##### 10. Recreate the Fixed Test Split

In [0]:
X_train, X_temp, y_train, y_temp = (
    train_test_split(
        X,
        y,
        test_size=TEST_SIZE,
        random_state=RANDOM_STATE,
        stratify=y,
    )
)

In [0]:
X_val, X_test, y_val, y_test = (
    train_test_split(
        X_temp,
        y_temp,
        test_size=VALIDATION_TEST_SIZE,
        random_state=RANDOM_STATE,
        stratify=y_temp,
    )
)

In [0]:
print(
    "Test features:",
    X_test.shape,
)

print(
    "Test targets:",
    y_test.shape,
)

##### 11. Transform Test Features

In [0]:
X_test_processed = (
    preprocessor.transform(
        X_test
    )
)

In [0]:
print(
    "Processed test shape:",
    X_test_processed.shape,
)

##### 12. Convert to PyTorch Tensors

In [0]:
X_test_tensor = torch.tensor(
    X_test_processed,
    dtype=torch.float32,
)

y_test_tensor = torch.tensor(
    y_test.to_numpy(),
    dtype=torch.float32,
).unsqueeze(1)

In [0]:
print(
    "X_test_tensor:",
    X_test_tensor.shape,
)

print(
    "y_test_tensor:",
    y_test_tensor.shape,
)

##### 13. Final Registered-Model Inference

In [0]:
candidate_model.eval()

with torch.no_grad():

    test_logits = (
        candidate_model(
            X_test_tensor
        )
    )

In [0]:
print(
    "Test logits:",
    test_logits.shape,
)

##### 14. Structural Acceptance Checks

In [0]:
expected_output_shape = (
    len(X_test_tensor),
    1,
)

In [0]:
output_shape_pass = (
    tuple(test_logits.shape)
    == expected_output_shape
)

In [0]:
finite_logits_pass = (
    torch.isfinite(
        test_logits
    )
    .all()
    .item()
)

In [0]:
probability_range_pass


In [0]:
test_probabilities = torch.sigmoid(
    test_logits
)

In [0]:
probability_range_pass = (
    (
        test_probabilities
        >= 0
    )
    &
    (
        test_probabilities
        <= 1
    )
).all().item()

In [0]:
print(
    "Output shape:",
    "PASS"
    if output_shape_pass
    else "FAIL",
)

print(
    "Finite logits:",
    "PASS"
    if finite_logits_pass
    else "FAIL",
)

print(
    "Probability range:",
    "PASS"
    if probability_range_pass
    else "FAIL",
)

##### 15. Calculate Final Test Loss

In [0]:
criterion = (
    nn.BCEWithLogitsLoss()
)

test_loss = criterion(
    test_logits,
    y_test_tensor,
)

In [0]:
print(
    "Test loss:",
    round(
        test_loss.item(),
        4,
    ),
)

##### 16. Create Final Predictions

In [0]:
test_predictions = (
    test_probabilities
    >= CLASSIFICATION_THRESHOLD
).float()

probability >= 0.50
→ Churn

probability < 0.50
→ No Churn

##### 17. Convert Results to NumPy

In [0]:
y_true = (
    y_test_tensor
    .cpu()
    .numpy()
    .ravel()
)

y_pred = (
    test_predictions
    .cpu()
    .numpy()
    .ravel()
)

y_prob = (
    test_probabilities
    .cpu()
    .numpy()
    .ravel()
)

#####  18. Calculate Final Acceptance Metrics

In [0]:
accuracy = accuracy_score(
    y_true,
    y_pred,
)

precision = precision_score(
    y_true,
    y_pred,
)

recall = recall_score(
    y_true,
    y_pred,
)

f1 = f1_score(
    y_true,
    y_pred,
)

roc_auc = roc_auc_score(
    y_true,
    y_prob,
)

In [0]:
print(
    "Accuracy:",
    round(
        accuracy,
        4,
    ),
)

print(
    "Precision:",
    round(
        precision,
        4,
    ),
)

print(
    "Recall:",
    round(
        recall,
        4,
    ),
)

print(
    "F1 Score:",
    round(
        f1,
        4,
    ),
)

print(
    "ROC-AUC:",
    round(
        roc_auc,
        4,
    ),
)

##### 19. Final Confusion Matrix

In [0]:
cm = confusion_matrix(
    y_true,
    y_pred,
)

tn, fp, fn, tp = (
    cm.ravel()
)

In [0]:
print(
    "True Negatives:",
    tn,
)

print(
    "False Positives:",
    fp,
)

print(
    "False Negatives:",
    fn,
)

print(
    "True Positives:",
    tp,
)

##### 20. Define Acceptance Checks

In [0]:
#Now compare actual results against the thresholds we defined before final evaluation:

acceptance_checks = {
    "output_shape":
        output_shape_pass,

    "finite_logits":
        finite_logits_pass,

    "probability_range":
        probability_range_pass,

    "accuracy":
        accuracy
        >=
        MIN_ACCEPTANCE_ACCURACY,

    "precision":
        precision
        >=
        MIN_ACCEPTANCE_PRECISION,

    "recall":
        recall
        >=
        MIN_ACCEPTANCE_RECALL,

    "f1_score":
        f1
        >=
        MIN_ACCEPTANCE_F1,

    "roc_auc":
        roc_auc
        >=
        MIN_ACCEPTANCE_ROC_AUC,
}

##### 21. Display Acceptance Results

In [0]:
for check, passed in (
    acceptance_checks.items()
):

    print(
        f"{check}: "
        f"{'PASS' if passed else 'FAIL'}"
    )

##### 22. Overall Acceptance Decision

In [0]:
overall_acceptance_pass = all(
    acceptance_checks.values()
)

In [0]:
print(
    "\nOverall acceptance:",
    (
        "PASS"
        if overall_acceptance_pass
        else "FAIL"
    ),
)

all checks true
       ↓
PASS


one or more checks false
       ↓
FAIL

##### 23. Log Acceptance Test to MLflow

This is a useful production improvement because final testing should also be auditable.

Define an acceptance experiment:

In [0]:
ACCEPTANCE_EXPERIMENT_PATH = (
    "/Users/sujathakrishna2811@gmail.com/"
    "telco_churn_nn_acceptance_tests"
)

mlflow.set_experiment(
    ACCEPTANCE_EXPERIMENT_PATH
)

In [0]:
with mlflow.start_run(
    run_name=(
        f"candidate_v"
        f"{CANDIDATE_VERSION}_acceptance"
    )
) as acceptance_run:

    mlflow.log_params(
        {
            "registered_model":
                REGISTERED_MODEL_NAME,

            "candidate_version":
                CANDIDATE_VERSION,

            "classification_threshold":
                CLASSIFICATION_THRESHOLD,

            "source_run_id":
                SELECTED_RUN_ID,

            "min_accuracy":
                MIN_ACCEPTANCE_ACCURACY,

            "min_precision":
                MIN_ACCEPTANCE_PRECISION,

            "min_recall":
                MIN_ACCEPTANCE_RECALL,

            "min_f1":
                MIN_ACCEPTANCE_F1,

            "min_roc_auc":
                MIN_ACCEPTANCE_ROC_AUC,
        }
    )

    mlflow.log_metrics(
        {
            "test_loss":
                test_loss.item(),

            "accuracy":
                accuracy,

            "precision":
                precision,

            "recall":
                recall,

            "f1_score":
                f1,

            "roc_auc":
                roc_auc,

            "true_negatives":
                int(tn),

            "false_positives":
                int(fp),

            "false_negatives":
                int(fn),

            "true_positives":
                int(tp),
        }
    )

    mlflow.set_tag(
        "acceptance_status",
        (
            "PASS"
            if overall_acceptance_pass
            else "FAIL"
        ),
    )

    ACCEPTANCE_RUN_ID = (
        acceptance_run.info.run_id
    )

In [0]:
print(
    "Acceptance Run ID:",
    ACCEPTANCE_RUN_ID,
)

##### 24. Promote Candidate to Champion Only If PASS

In [0]:
if overall_acceptance_pass:

    client.set_registered_model_alias(
        name=REGISTERED_MODEL_NAME,
        alias=CHAMPION_ALIAS,
        version=CANDIDATE_VERSION,
    )

    print(
        "Candidate promoted to Champion."
    )

else:

    print(
        "Candidate was NOT promoted "
        "to Champion."
    )

##### 25. Update Model Version Tags

In [0]:
client.set_model_version_tag(
    name=REGISTERED_MODEL_NAME,
    version=CANDIDATE_VERSION,
    key="acceptance_status",
    value=(
        "passed"
        if overall_acceptance_pass
        else "failed"
    ),
)

In [0]:
client.set_model_version_tag(
    name=REGISTERED_MODEL_NAME,
    version=CANDIDATE_VERSION,
    key="acceptance_run_id",
    value=ACCEPTANCE_RUN_ID,
)

In [0]:
if overall_acceptance_pass:

    client.set_model_version_tag(
        name=REGISTERED_MODEL_NAME,
        version=CANDIDATE_VERSION,
        key="lifecycle_status",
        value="champion",
    )

##### 26. Verify Champion Alias

In [0]:
# only when passed
if overall_acceptance_pass:

    champion_version = (
        client.get_model_version_by_alias(
            name=REGISTERED_MODEL_NAME,
            alias=CHAMPION_ALIAS,
        )
    )

    print(
        "Champion version:",
        champion_version.version,
    )

##### 27. Build Final Project Summary

In [0]:
final_project_summary = {
    "registered_model":
        REGISTERED_MODEL_NAME,

    "candidate_version":
        CANDIDATE_VERSION,

    "acceptance_status":
        (
            "PASS"
            if overall_acceptance_pass
            else "FAIL"
        ),

    "accuracy":
        accuracy,

    "precision":
        precision,

    "recall":
        recall,

    "f1_score":
        f1,

    "roc_auc":
        roc_auc,

    "test_loss":
        test_loss.item(),

    "source_run_id":
        SELECTED_RUN_ID,

    "acceptance_run_id":
        ACCEPTANCE_RUN_ID,

    "champion_promoted":
        overall_acceptance_pass,
}

In [0]:
for key, value in (
    final_project_summary.items()
):

    print(
        f"{key}: {value}"
    )

telco_churn_nn_experiments
        ↓
Training / hyperparameter experiments


telco_churn_nn_acceptance_tests
        ↓
Final registered-model validation

##### Key Learnings

1. Final acceptance testing is separate from model training and model selection.

2. The registered Candidate is loaded directly from Unity Catalog.

3. Final testing does not depend on notebook-memory model objects.

4. The same fitted preprocessing pipeline used for model training must be used for final inference.

5. Final acceptance testing uses transform(), never fit_transform().

6. model.eval() and torch.no_grad() are used for inference.

7. Structural validation should occur before evaluating predictive performance.

8. Output shape validation confirms that the registered model produces one logit per customer.

9. Finite-value checks help detect invalid model outputs.

10. Sigmoid converts logits into probabilities between 0 and 1.

11. Classification thresholds convert probabilities into classes.

12. Threshold tuning should not occur during final acceptance testing.

13. y_test provides the ground-truth labels used to calculate final model metrics.

14. Accuracy measures overall correctness.

15. Precision measures how reliable Churn predictions are.

16. Recall measures how many actual churners are detected.

17. F1 balances precision and recall.

18. ROC-AUC measures ranking/discrimination ability across thresholds.

19. Acceptance criteria should be established before inspecting final acceptance results.

20. A Candidate passes only if every mandatory acceptance criterion passes.

21. MLflow provides an auditable record of final acceptance testing.

22. Candidate and Champion represent different lifecycle roles.

23. Candidate identifies the version undergoing final validation.

24. Champion identifies the approved model version.

25. Champion promotion can be automated using the Model Registry API.

26. A failed Candidate should not replace the existing Champion.

27. The Candidate and Champion aliases may temporarily point to the same version after successful promotion.

28. Future model versions can become Candidate while the existing Champion remains unchanged.

29. Registration, acceptance testing, and production promotion are intentionally separate lifecycle stages.

30. Production ML is not only about training a model; it is about reproducibility, validation, governance, lineage, controlled promotion, and monitoring.

##### Conclusion

This notebook completed the production-style Telco Churn neural-network lifecycle.

The registered Candidate model was loaded independently from Unity Catalog.

The exact fitted preprocessing pipeline was loaded from persistent storage and used to transform the fixed test dataset.

The Candidate model was evaluated using:

- structural checks,
- test loss,
- accuracy,
- precision,
- recall,
- F1 Score,
- ROC-AUC,
- and confusion-matrix results.

The results were compared against predefined acceptance criteria.

The complete acceptance test was logged to MLflow for traceability.

If all required checks passed, the registered Candidate version was automatically assigned the Champion alias.

No retraining or hyperparameter tuning occurred during final acceptance testing.

The lifecycle implemented throughout this project was:


``` text

Data Preparation
        ↓
Neural-Network Architecture
        ↓
Training
        ↓
Validation
        ↓
Checkpointing
        ↓
Early Stopping
        ↓
MLflow Tracking
        ↓
Test Evaluation
        ↓
ML vs DL Comparison
        ↓
Controlled NN Experiments
        ↓
Candidate Selection
        ↓
Model Registration
        ↓
Candidate Alias
        ↓
Final Acceptance Testing
        ↓
Champion Promotion

```

The project demonstrates both the mechanics of neural-network learning and the production lifecycle required to manage a trained model reliably.